# Prompt Optimization with GEPA

This notebook demonstrates how to use Training Hub's GEPA (Genetic-Pareto) algorithm to optimize textual prompts **without modifying model weights**. GEPA uses evolutionary search with Pareto-based selection and LLM-driven reflection to evolve prompts that maximize task performance.

We'll cover both backends:
1. **GEPA backend** (default) — calls `gepa.optimize()` directly
2. **MLflow backend** — uses `mlflow.genai.optimize_prompts()` for integration with MLflow's prompt registry and experiment tracking

## What makes GEPA different?

Unlike weight-training algorithms (SFT, OSFT, LoRA, GRPO), GEPA optimizes the **prompt itself**:
- No GPU required for training (only for serving the model)
- No model weights are modified
- Useful for improving system prompts, few-shot templates, and agent instructions
- Works with any LLM accessible via an API endpoint

## Prerequisites

- A running vLLM server serving a small model (setup instructions below)
- `training-hub[gepa]` installed

## Setup

### Step 1: Start a local vLLM server

GEPA needs an LLM endpoint for both task evaluation and prompt reflection. The **reflection model** must be capable of meta-reasoning (analyzing *why* prompts succeed or fail and proposing better ones), so it needs to be at least 3B parameters. Smaller models (0.5B–1.5B) produce incoherent reflections — they output JSON task descriptions instead of system prompts, and the seed prompt never improves.

In a **separate terminal**, run:

```bash
# Install vLLM if not already installed
pip install vllm

# Serve a model suitable for both task evaluation and reflection.
# Qwen2.5-3B-Instruct is the smallest model that reflects well (~6GB VRAM).
vllm serve Qwen/Qwen2.5-3B-Instruct \
    --max-model-len 2048 \
    --gpu-memory-utilization 0.5

# For better reflection quality (recommended if you have the VRAM):
# vllm serve Qwen/Qwen2.5-7B-Instruct --max-model-len 2048 --gpu-memory-utilization 0.8
```

Wait until you see `Application startup complete` before proceeding.

### Step 2: Install dependencies

In [ ]:
# Install training-hub with GEPA dependencies (includes gepa, litellm, mlflow)
%pip install 'training-hub[gepa]' --quiet

In [ ]:
import json
import os
import warnings
from pathlib import Path

# Suppress Pydantic serialization warnings from litellm/vLLM response objects.
# These are version-mismatch noise, not real errors.
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

# Configuration — adjust these to match your vLLM server
VLLM_BASE_URL = "http://localhost:8000/v1"
MODEL_NAME = "openai/Qwen/Qwen2.5-3B-Instruct"  # litellm format: openai/<model>
OUTPUT_DIR = Path("./gepa_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# litellm requires an API key even for local vLLM endpoints.
# Set a dummy value if no real key is configured — vLLM ignores this,
# but litellm refuses to send requests without one.
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY") or "dummy"

print(f"vLLM endpoint: {VLLM_BASE_URL}")
print(f"Model: {MODEL_NAME}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Verify the vLLM server is running
import litellm

try:
    response = litellm.completion(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": "Say hello in one word."}],
        api_base=VLLM_BASE_URL,
        max_tokens=16,
    )
    print(f"Server is running. Test response: {response.choices[0].message.content}")
except Exception as e:
    print(f"ERROR: Cannot reach vLLM server at {VLLM_BASE_URL}")
    print(f"Details: {e}")
    print("\nMake sure vLLM is running with a 3B+ model (see instructions above).")

## Prepare Training Data

GEPA expects data in JSONL format with `input` and `answer` fields. We'll create a focused dataset of 10 questions designed so that a **vague system prompt** causes the model to fail, but a **well-crafted prompt** can guide it to the exact expected answer.

The key insight: GEPA's default evaluator (`ContainsAnswerEvaluator`) checks whether the expected answer string appears in the model's response. A model with a vague prompt will:
- Add extra reasoning, caveats, or hedging that buries the answer
- Give approximate answers instead of exact ones (e.g., "around 200" instead of "206")
- Use full sentences when a terse answer is needed
- Elaborate when the question requires a single word or number

A well-optimized prompt like *"Answer with only the exact value, no explanation"* should fix these issues.

**Budget rule of thumb:** `max_metric_calls >= 20 * dataset_size` to give GEPA enough room for 15-20 iterations of exploration. With 10 examples and `max_metric_calls=200`, GEPA gets ~20 iterations.

In [ ]:
# Create a focused training dataset where a vague prompt fails but a precise one succeeds.
# Each question requires an EXACT short answer — models with vague prompts tend to
# elaborate, hedge, or give approximate answers that don't contain the exact string.
train_data = [
    # Requires exact numeric output (model tends to say "approximately" or add units)
    {"input": "Express 3/4 as a decimal.", "answer": "0.75"},
    {"input": "What is the square root of 144?", "answer": "12"},
    {"input": "What is 15% of 200?", "answer": "30"},
    # Requires single-word precision (model tends to add explanation)
    {"input": "What is the pH of pure water at 25°C? Give only the number.", "answer": "7"},
    {"input": "How many sides does a hexagon have? Answer with just the number.", "answer": "6"},
    # Requires specific format (model may use synonyms or elaborate)
    {"input": "What is the chemical symbol for gold?", "answer": "Au"},
    {"input": "What is the SI unit of electric current?", "answer": "ampere"},
    # Requires disambiguation (model may give alternate valid answers)
    {"input": "Name the three states of matter, separated by commas.", "answer": "solid, liquid, gas"},
    {"input": "What is the Roman numeral for 50?", "answer": "L"},
    # Requires constrained output (model tends to over-explain)
    {"input": "Is zero even or odd? Answer with one word.", "answer": "even"},
]

# Save to JSONL
data_path = OUTPUT_DIR / "train_data.jsonl"
with open(data_path, "w") as f:
    for entry in train_data:
        f.write(json.dumps(entry) + "\n")

print(f"Training data saved to {data_path} ({len(train_data)} examples)")
print(f"\nSample entries:")
for entry in train_data[:3]:
    print(f"  Q: {entry['input']}")
    print(f"  A: {entry['answer']}")

## Part 1: GEPA Backend (Default)

The default GEPA backend calls `gepa.optimize()` directly. It evolves a `seed_candidate` prompt through mutations guided by LLM reflection, selecting candidates using Pareto-based strategies.

Key parameters:
- **`seed_candidate`**: The starting prompt to optimize (dict of field names to text)
- **`task_lm`**: The model to evaluate prompts against (litellm format)
- **`api_base`**: Points litellm at your local vLLM server
- **`max_metric_calls`**: Budget — how many evaluation calls GEPA can make. Use at least `20 * dataset_size` for meaningful optimization.
- **`reflection_minibatch_size`**: How many examples GEPA examines per reflection step. Higher values help when baseline accuracy is high, ensuring failing examples are included in each reflection batch.

In [ ]:
from training_hub import gepa

# Start with a basic prompt that GEPA will improve.
# This deliberately vague prompt gives the model no guidance on format,
# precision, or conciseness — GEPA should evolve something better.
seed_prompt = {"system_prompt": "Answer the question."}

gepa_result = gepa(
    seed_candidate=seed_prompt,
    task_lm=MODEL_NAME,
    api_base=VLLM_BASE_URL,
    data_path=str(data_path),
    max_metric_calls=200,  # ~20 iterations with 10 examples — enough for real improvement
    reflection_minibatch_size=5,  # examine more examples per reflection to catch failures
    output_dir=str(OUTPUT_DIR / "gepa_backend"),
)

print("\n" + "=" * 60)
print("Optimization complete!")
print("=" * 60)

In [ ]:
# Inspect the result
print("Best prompt found:")
print("-" * 40)
for field, text in gepa_result.best_candidate.items():
    print(f"  {field}: {text}")

# GEPAResult stores scores in val_aggregate_scores (indexed by candidate)
best_score = gepa_result.val_aggregate_scores[gepa_result.best_idx]
print(f"\nBest score: {best_score:.2f}")
print(f"Candidates explored: {gepa_result.num_candidates}")
print(f"\nStarting prompt was: {seed_prompt}")

In [ ]:
# Check saved artifacts
saved_dir = OUTPUT_DIR / "gepa_backend"
print("Saved artifacts:")
for f in sorted(saved_dir.iterdir()):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

# Load and display the best candidate
with open(saved_dir / "best_candidate.json") as f:
    saved = json.load(f)
print(f"\nSaved best_candidate.json: {json.dumps(saved, indent=2)}")

## Part 2: MLflow Backend

The MLflow backend wraps GEPA with MLflow's prompt registry, scorer framework, and experiment tracking via `mlflow.genai.optimize_prompts()`. This is useful for teams already using MLflow for experiment management.

### MLflow setup

The MLflow backend requires:
1. An MLflow tracking server (we'll use a local one)
2. A **registered prompt** in MLflow's prompt registry
3. A **predict function** that uses the registered prompt
4. **Scorers** to evaluate prompt quality

> **Note on built-in scorers:** MLflow's built-in scorers like `Correctness` hardcode the OpenAI API endpoint and don't route through `OPENAI_API_BASE`. For local models, use custom `@scorer` functions with litellm instead.

In [ ]:
import mlflow
from mlflow.genai.scorers import scorer

# Start a local MLflow tracking server (uses local file store)
mlflow_dir = OUTPUT_DIR / "mlflow_tracking"
mlflow_dir.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"file://{mlflow_dir.resolve()}")
mlflow.set_experiment("gepa-prompt-optimization")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"MLflow experiment: gepa-prompt-optimization")

In [ ]:
# Register a prompt template in MLflow's prompt registry
prompt = mlflow.genai.register_prompt(
    name="qa_system_prompt",
    template="{{ system_prompt }}\n\nQuestion: {{ input }}\nAnswer:",
)

print(f"Registered prompt: {prompt.name}")
print(f"Prompt URI: {prompt.uri}")
print(f"Template: {prompt.template}")

In [ ]:
# Define the predict function — this is called by MLflow during optimization.
# MLflow unpacks each data entry's "inputs" dict and passes the keys as keyword
# arguments, so the parameter name must match the data key (here: "input").
def predict_fn(input) -> str:
    """Generate a response using the registered prompt and local vLLM."""
    question = input

    response = litellm.completion(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        api_base=VLLM_BASE_URL,
        max_tokens=64,
    )
    return response.choices[0].message.content


# Define a custom scorer that works with local models.
# This checks if the expected answer appears in the model's response.
@scorer
def answer_contains(inputs, outputs, expectations):
    """Score 1.0 if the expected answer appears in the output, else 0.0."""
    import re
    expected = (expectations or {}).get("expected_response", "")
    # Strip any think blocks (for models that use chain-of-thought)
    clean_output = re.sub(r"<think>.*?</think>", "", outputs or "", flags=re.DOTALL).strip()
    return 1.0 if expected.lower() in clean_output.lower() else 0.0


print("predict_fn and scorer defined.")

In [ ]:
# Prepare data in MLflow format:
# MLflow expects {"inputs": {...}, "expectations": {...}} dicts.
mlflow_train_data = [
    {
        "inputs": {"input": entry["input"]},
        "expectations": {"expected_response": entry["answer"]},
    }
    for entry in train_data
]

print(f"MLflow training data: {len(mlflow_train_data)} examples")
print(f"\nSample entries:")
for entry in mlflow_train_data[:2]:
    print(f"  {json.dumps(entry)}")

In [ ]:
# Run GEPA optimization via the MLflow backend
mlflow_result = gepa(
    seed_candidate={"system_prompt": "Answer the question."},
    task_lm=MODEL_NAME,
    api_base=VLLM_BASE_URL,
    backend="mlflow",
    # MLflow-specific parameters
    predict_fn=predict_fn,
    prompt_uris=[prompt.uri],
    scorers=[answer_contains],
    trainset=mlflow_train_data,  # Pass pre-formatted MLflow data directly
    enable_tracking=False,  # Disable MLflow trace tracking to reduce overhead
    # GEPA optimization config
    max_metric_calls=200,  # Match GEPA backend budget for fair comparison
    reflection_minibatch_size=5,  # examine more examples per reflection
    output_dir=str(OUTPUT_DIR / "mlflow_backend"),
)

print("\n" + "=" * 60)
print("MLflow optimization complete!")
print("=" * 60)

In [ ]:
# Inspect MLflow results
print(f"Optimizer: {mlflow_result.optimizer_name}")
print(f"Initial score: {mlflow_result.initial_eval_score:.2f}")
print(f"Final score:   {mlflow_result.final_eval_score:.2f}")
print(f"\nOptimized prompts:")
for p in mlflow_result.optimized_prompts:
    print(f"  URI: {p.uri}")

In [ ]:
# View the optimized prompt registered in MLflow
optimized_prompt = mlflow.genai.get_prompt("qa_system_prompt")
print(f"Latest registered prompt version: {optimized_prompt.version}")
print(f"Template:\n{optimized_prompt.template}")

## Comparing the Two Backends

| | GEPA Backend | MLflow Backend |
|--|-------------|----------------|
| **Best for** | Standalone use, quick experiments | Teams using MLflow for experiment tracking |
| **Prompt storage** | Saved to `best_candidate.json` | Registered in MLflow prompt registry |
| **Scoring** | Built-in `ContainsAnswerEvaluator` | Custom `@scorer` functions or MLflow built-in scorers |
| **Tracking** | Optional (W&B, MLflow via GEPA) | Native MLflow experiment tracking |
| **Setup** | Minimal | Requires MLflow tracking server + prompt registration |
| **Local models** | Via `api_base` | Via `api_base` + custom scorers (built-in scorers hardcode OpenAI) |

## Tips for Real Usage

### Choosing `max_metric_calls`
A good rule of thumb is `max_metric_calls >= 20 * dataset_size`:
- **10 examples, budget 200**: ~20 iterations — solid for demos and quick experiments
- **10 examples, budget 300**: ~30 iterations — thorough exploration
- **20 examples, budget 400-500**: thorough exploration for production prompts

### Model size for reflection
The `reflection_lm` must be capable of meta-reasoning about prompt quality. Models under 3B parameters typically produce incoherent reflections — they output JSON task descriptions or nonsensical text instead of improved system prompts. Recommendations:
- **Minimum**: 3B parameters (e.g., `Qwen/Qwen2.5-3B-Instruct`)
- **Recommended**: 7B+ for complex tasks
- **Split setup**: Use a small model as `task_lm` and a larger model as `reflection_lm` — this tests whether prompts optimized by a strong model transfer to a weak one

```python
# Split setup: optimize prompts for a small model using a larger reflector
result = gepa(
    seed_candidate={"system_prompt": "..."},
    task_lm="openai/Qwen/Qwen2.5-1.5B-Instruct",
    reflection_lm="openai/Qwen/Qwen2.5-7B-Instruct",
    api_base="http://localhost:8000/v1",  # Both models on same server
    ...
)
```

### Designing good training data
GEPA works best when there's a clear gap between a vague prompt and a good one:
- **Require exact formatting**: "Express as a decimal" → "0.75" (model might say "three quarters")
- **Constrain output length**: "Answer with one word" → "even" (model might elaborate)
- **Demand precision**: "Give only the number" → "7" (model might say "approximately 7")

A baseline accuracy of ~0.3-0.5 with a vague prompt gives GEPA enough room to improve. If baseline is already 0.8+, there's too little signal for the reflection model to learn from.

### Reflection minibatch size
Set `reflection_minibatch_size=5` or higher (default is 2-3). With a small default, GEPA only examines 2-3 examples when deciding if a candidate is worth exploring. If most examples pass (high baseline), the subsample may contain only passing examples and GEPA skips the reflection step entirely.

### Using larger models
GEPA works with any model accessible via litellm. To use a larger local model:
```python
# Serve a larger model
# vllm serve Qwen/Qwen2.5-7B-Instruct --max-model-len 4096 --gpu-memory-utilization 0.8

result = gepa(
    seed_candidate={"system_prompt": "..."},
    task_lm="openai/Qwen/Qwen2.5-7B-Instruct",
    api_base="http://localhost:8000/v1",
    ...
)
```

### Custom evaluators (GEPA backend)
For tasks beyond simple string matching, pass a custom `evaluator`:
```python
def my_evaluator(data, response):
    # score: float, feedback: str, objective_scores: dict
    score = compute_my_metric(data["answer"], response)
    return score, f"Score: {score}", {"accuracy": score}

result = gepa(
    ...,
    evaluator=my_evaluator,
)
```

## Cleanup

In [ ]:
# Uncomment to remove output files
# import shutil
# shutil.rmtree(OUTPUT_DIR)
# print(f"Removed {OUTPUT_DIR}")